# Pixels to Predictions — SmolVLM-500M Starter

## 0. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/data

Mounted at /content/drive
images.zip  sample_submission.csv  test.csv  train.csv	val.csv


## 1. Install packages

In [2]:
!pip install -q -U transformers==4.49.0 peft==0.13.2 accelerate==1.0.1
!pip install -q pillow pandas tqdm num2words safetensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 7.4 MB/s eta 0:00:00


In [3]:
import transformers, peft
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)

import importlib.util
print("bitsandbytes installed:", importlib.util.find_spec("bitsandbytes") is not None)

from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("idefics3 in CONFIG_MAPPING:", "idefics3" in CONFIG_MAPPING)

transformers: 4.49.0
peft: 0.13.2
bitsandbytes installed: False
idefics3 in CONFIG_MAPPING: True


## 2. Imports, seeds, paths

In [4]:
import os, json, random, shutil, zipfile, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

from transformers import AutoProcessor, AutoModelForVision2Seq, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "-")

Device: cuda | GPU: Tesla T4


In [5]:
VERSION = "v10"   # change this in each copy: "v1", "v2", "v3", ...

# Drive locations
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_DRIVE = DRIVE_ROOT / "data"
IMG_ZIP    = DATA_DRIVE / "images.zip"

# Local cache
LOCAL_ROOT     = Path("/content")
LOCAL_IMG_ZIP  = LOCAL_ROOT / "images.zip"
LOCAL_IMG_ROOT = LOCAL_ROOT
SENTINEL       = LOCAL_ROOT / ".images_extracted"

OUT_DIR = DRIVE_ROOT / VERSION / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [DATA_DRIVE/"train.csv", DATA_DRIVE/"val.csv", DATA_DRIVE/"test.csv",
          DATA_DRIVE/"sample_submission.csv", IMG_ZIP]:
    assert p.exists(), f"Missing on Drive: {p}"

CONFIG = dict(
    model_id            = "HuggingFaceTB/SmolVLM-500M-Instruct",
    image_longest_edge  = 384,
    lora_r              = 24,
    lora_alpha          = 48,
    lora_dropout        = 0.05,
    epochs              = 5,
    lr                  = 1e-4,
    batch_size          = 2,
    grad_accum          = 8,
    warmup_ratio        = 0.03,
    max_text_tokens     = 1280,
    use_cot             = True,
    cot_prob            = 0.5,
    val_every_steps     = 50,
    save_path           = str(OUT_DIR / "lora_adapter"),
    ckpt_latest_path    = str(OUT_DIR / "ckpt_latest"),
    ckpt_every_steps    = 50,
    # NEW for v10: raw <image> prompt format (no chat template)
    prompt_style        = "raw_image",
    use_choice_aug      = False,
    # Reverted from v9 — train+val combined hurt test, drop it
    use_train_plus_val  = False,
    # Keep these — they didn't appear to be the v9 problem
    label_smoothing     = 0.1,
    use_ema             = True,
    ema_decay           = 0.999,
    tta_n_permutations  = 3,
)

print(f"VERSION = {VERSION}, prompt_style = {CONFIG['prompt_style']}")
print(f"Outputs: {OUT_DIR}")

VERSION = v10, prompt_style = raw_image
Outputs: /content/drive/MyDrive/v10/outputs


## 3. Copy + extract `images.zip` to local (cached)

Reading thousands of small PNGs straight from Drive is ~100× slower than local disk — every file is a separate network call. This cell copies `images.zip` from Drive to `/content/` and extracts once. A sentinel `/content/.images_extracted` makes re-runs within the same Colab session a no-op. When the VM recycles (~12h or on disconnect), this runs again from scratch (2–5 min).

In [6]:
def ensure_local_images():
    if SENTINEL.exists():
        print(f"Images already extracted at {LOCAL_IMG_ROOT/'images'} — skipping.")
        return
    if not LOCAL_IMG_ZIP.exists():
        size_mb = IMG_ZIP.stat().st_size / 1e6
        print(f"Copying {IMG_ZIP.name} ({size_mb:.0f} MB) from Drive -> /content/ ...")
        t0 = time.time()
        shutil.copy(IMG_ZIP, LOCAL_IMG_ZIP)
        print(f"  copy done in {time.time()-t0:.1f}s")
    print("Extracting ...")
    t0 = time.time()
    with zipfile.ZipFile(LOCAL_IMG_ZIP) as z:
        z.extractall(LOCAL_IMG_ROOT)
    print(f"  extract done in {time.time()-t0:.1f}s")
    for split in ("train", "val", "test"):
        d = LOCAL_IMG_ROOT / "images" / split
        assert d.exists(), (
            f"Expected {d} after extract. Did you zip with `images/` as the top-level dir? "
            f"From the parent of `images/`, run: zip -r images.zip images/"
        )
        n = sum(1 for _ in d.glob("*.png"))
        print(f"  images/{split}: {n} PNGs")
    SENTINEL.touch()
    LOCAL_IMG_ZIP.unlink(missing_ok=True)   # free disk; sentinel protects against re-extract

ensure_local_images()

Copying images.zip (375 MB) from Drive -> /content/ ...
  copy done in 14.1s
Extracting ...
  extract done in 2.2s
  images/train: 3109 PNGs
  images/val: 1048 PNGs
  images/test: 1008 PNGs


## 4. Load CSVs (from Drive) and wire image paths to local extraction

In [7]:
def load_split(name):
    df = pd.read_csv(DATA_DRIVE / f"{name}.csv")
    df["choices"] = df["choices"].apply(json.loads)
    df["image_abs"] = df["image_path"].apply(lambda p: str(LOCAL_IMG_ROOT / p))
    return df

train_df = load_split("train")
val_df   = load_split("val")
test_df  = load_split("test")

if CONFIG["use_train_plus_val"]:
    print(f"Combining train ({len(train_df)}) + val ({len(val_df)}) for training")
    train_df_full = pd.concat([train_df, val_df], ignore_index=True)
else:
    train_df_full = train_df
print(f"Effective training set: {len(train_df_full)} rows")

Effective training set: 3109 rows


## 5. Load SmolVLM + attach LoRA (language-model only)

In [8]:
processor = AutoProcessor.from_pretrained(CONFIG["model_id"])
if hasattr(processor, "image_processor"):
    processor.image_processor.size = {"longest_edge": CONFIG["image_longest_edge"]}

model = AutoModelForVision2Seq.from_pretrained(
    CONFIG["model_id"], torch_dtype=torch.float16, low_cpu_mem_usage=True,
).to(DEVICE)

for p in model.parameters():
    p.requires_grad = False

target_modules = []
for name, _ in model.named_modules():
    if any(name.endswith(f".{proj}") for proj in ("q_proj","k_proj","v_proj","o_proj")):
        if any(tag in name for tag in ("text_model","language_model","text_decoder")):
            target_modules.append(name)

if not target_modules:
    for name, _ in model.named_modules():
        if any(name.endswith(f".{proj}") for proj in ("q_proj","k_proj","v_proj","o_proj")):
            if "vision" not in name and "visual" not in name:
                target_modules.append(name)

assert target_modules, "No LoRA target modules found — inspect model.named_modules()"
print(f"LoRA adapting {len(target_modules)} modules. Samples:")
for n in target_modules[:4]: print(" ", n)

lora_cfg = LoraConfig(
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
    bias="none", task_type=TaskType.CAUSAL_LM, target_modules=target_modules,
)
model = get_peft_model(model, lora_cfg)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,}  ({trainable/total:.2%} of {total:,})")
assert trainable <= 5_000_000, f"OVER 5M CAP: {trainable:,}"
print("Under 5M cap ✓")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

LoRA adapting 128 modules. Samples:
  model.text_model.layers.0.self_attn.q_proj
  model.text_model.layers.0.self_attn.k_proj
  model.text_model.layers.0.self_attn.v_proj
  model.text_model.layers.0.self_attn.o_proj
Trainable: 4,915,200  (0.96% of 512,397,504)
Under 5M cap ✓


## 6. Prompt + letter-token scoring

In [9]:
LETTERS = ["A", "B", "C", "D", "E"]

def resolve_letter_token_ids(tokenizer, letters=LETTERS):
    ids = []
    for L in letters:
        tok_id = None
        for candidate in (f" {L}", L, f"\n{L}"):
            enc = tokenizer.encode(candidate, add_special_tokens=False)
            if len(enc) == 1:
                tok_id = enc[0]; break
        if tok_id is None:
            tok_id = tokenizer.encode(f" {L}", add_special_tokens=False)[-1]
        ids.append(tok_id)
    return ids

LETTER_IDS = resolve_letter_token_ids(processor.tokenizer)
print("Letter token IDs:", dict(zip(LETTERS, LETTER_IDS)))


def _choice_lines(row):
    return "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(row["choices"]))


def _build_v5_baseline(row, include_solution):
    parts = []
    if isinstance(row.get("lecture"), str) and row["lecture"].strip():
        parts.append(f"Lecture: {row['lecture'].strip()}")
    if isinstance(row.get("hint"), str) and row["hint"].strip():
        parts.append(f"Context: {row['hint'].strip()}")
    parts.append(f"Question: {row['question'].strip()}")
    parts.append("Choices:\n" + _choice_lines(row))
    if include_solution and isinstance(row.get("solution"), str) and row["solution"].strip():
        parts.append(f"Reasoning hint: {row['solution'].strip()}")
    valid = "".join(LETTERS[: int(row["num_choices"])])
    parts.append(f"Respond with a single letter from [{', '.join(valid)}].")
    parts.append("Answer:")
    return "\n\n".join(parts)


def _build_compact(row, include_solution):
    parts = []
    if isinstance(row.get("lecture"), str) and row["lecture"].strip():
        parts.append(row["lecture"].strip())
    if isinstance(row.get("hint"), str) and row["hint"].strip():
        parts.append(row["hint"].strip())
    parts.append(row["question"].strip())
    parts.append(_choice_lines(row))
    if include_solution and isinstance(row.get("solution"), str) and row["solution"].strip():
        parts.append(row["solution"].strip())
    parts.append("Answer:")
    return "\n\n".join(parts)


def _build_compact_qfirst(row, include_solution):
    parts = []
    parts.append(row["question"].strip())
    parts.append(_choice_lines(row))
    extras = []
    if isinstance(row.get("hint"), str) and row["hint"].strip():
        extras.append(row["hint"].strip())
    if isinstance(row.get("lecture"), str) and row["lecture"].strip():
        extras.append(row["lecture"].strip())
    if extras:
        parts.append("\n\n".join(extras))
    if include_solution and isinstance(row.get("solution"), str) and row["solution"].strip():
        parts.append(row["solution"].strip())
    parts.append("Answer:")
    return "\n\n".join(parts)


def _build_raw_image(row, include_solution):
    """Starter-notebook style: bare <image>...Answer: with no chat template wrapper."""
    parts = []
    if isinstance(row.get("lecture"), str) and row["lecture"].strip():
        parts.append(row["lecture"].strip())
    if isinstance(row.get("hint"), str) and row["hint"].strip():
        parts.append(row["hint"].strip())

    text = "<image>\n"
    if parts:
        text += "Context:\n" + "\n".join(parts) + "\n\n"
    text += f"Question: {row['question'].strip()}\n"
    text += "Choices:\n"
    text += "\n".join(f"  {LETTERS[i]}. {c}" for i, c in enumerate(row["choices"]))
    text += "\n"
    if include_solution and isinstance(row.get("solution"), str) and row["solution"].strip():
        text += f"Reasoning: {row['solution'].strip()}\n"
    text += "Answer:"
    return text


_BUILDERS = {
    "v5_baseline":     _build_v5_baseline,
    "compact":         _build_compact,
    "compact_qfirst":  _build_compact_qfirst,
    "raw_image":       _build_raw_image,    # NEW
}


def build_user_text(row, include_solution=False):
    style = CONFIG.get("prompt_style", "v5_baseline")
    if style not in _BUILDERS:
        raise ValueError(f"unknown prompt_style: {style}")
    return _BUILDERS[style](row, include_solution)


def build_prompt(row, include_solution=False):
    """For raw_image style, skip the chat template (the builder emits the full text including <image>).
    For other styles, wrap in chat template as before."""
    style = CONFIG.get("prompt_style", "v5_baseline")
    if style == "raw_image":
        return build_user_text(row, include_solution=include_solution)
    messages = [{"role":"user", "content":[
        {"type":"image"},
        {"type":"text","text": build_user_text(row, include_solution=include_solution)},
    ]}]
    return processor.apply_chat_template(messages, add_generation_prompt=True)


# show all four formats on the same example, side-by-side
ex = train_df.iloc[0].to_dict()
print(f"\n=== ACTIVE STYLE: {CONFIG['prompt_style']} ===\n")
print(build_prompt(ex, include_solution=False)[:1200])
print("\n... (truncated)")

print("\n\n=== STYLE COMPARISON (first 400 chars of user text only) ===")
for style in ["v5_baseline", "compact", "compact_qfirst", "raw_image"]:
    print(f"\n--- {style} ---")
    text = _BUILDERS[style](ex, include_solution=False)
    print(text[:400])
    print(f"  ...total length: {len(text)} chars")

Letter token IDs: {'A': 330, 'B': 389, 'C': 340, 'D': 422, 'E': 414}

=== ACTIVE STYLE: raw_image ===

<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals wi

## 7. Dataset / collators

In [10]:
class VQADataset(Dataset):
    def __init__(self, df, mode, cot_prob=0.0):
        self.df = df.reset_index(drop=True)
        self.mode = mode
        self.cot_prob = cot_prob
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx].to_dict()
        img = Image.open(row["image_abs"]).convert("RGB")

        # Choice-order augmentation, controlled from CONFIG (off by default in v6)
        if (self.mode == "train"
                and CONFIG.get("use_choice_aug", False)
                and random.random() < 0.5):
            choices = list(row["choices"])
            n = len(choices)
            perm = list(range(n))
            random.shuffle(perm)
            row["choices"] = [choices[p] for p in perm]
            row["answer"]  = perm.index(row["answer"])

        include_sol = (self.mode=="train" and self.cot_prob>0
                    and isinstance(row.get("solution"), str)
                    and random.random() < self.cot_prob)
        prompt = build_prompt(row, include_solution=include_sol)
        item = dict(image=img, prompt=prompt, num_choices=int(row["num_choices"]), id=row["id"])
        if self.mode=="train": item["answer"] = int(row["answer"])
        return item

def collate_train(batch):
    images  = [b["image"] for b in batch]
    prompts = [b["prompt"] for b in batch]
    answers = [b["answer"] for b in batch]
    full_texts = [p + " " + LETTERS[a] for p, a in zip(prompts, answers)]
    enc = processor(text=full_texts, images=images, return_tensors="pt",
                    padding=True, truncation=True, max_length=CONFIG["max_text_tokens"])
    input_ids = enc["input_ids"]
    attn = enc["attention_mask"]
    last_idx = attn.sum(dim=1) - 1   # (B,) position of letter token

    # Build full labels with -100 mask everywhere except the last position
    labels = input_ids.clone(); labels[:] = -100
    for i, pos in enumerate(last_idx.tolist()):
        labels[i, pos] = input_ids[i, pos]
    enc["labels"] = labels
    # Pass these along so the train step can compute label-smoothed loss itself
    enc["_letter_pos"] = last_idx
    enc["_letter_id"]  = torch.tensor(
        [input_ids[i, p].item() for i, p in enumerate(last_idx.tolist())])
    return enc

def collate_infer(batch):
    images  = [b["image"] for b in batch]
    prompts = [b["prompt"] for b in batch]
    num_ch  = torch.tensor([b["num_choices"] for b in batch], dtype=torch.long)
    ids     = [b["id"] for b in batch]
    enc = processor(text=prompts, images=images, return_tensors="pt",
                    padding=True, truncation=True, max_length=CONFIG["max_text_tokens"])
    enc["num_choices"] = num_ch
    enc["ids"] = ids
    return enc

## 8. Inference / eval helpers

In [11]:
@torch.no_grad()
def predict(model, df, batch_size=4):
    model.eval()
    ds = VQADataset(df, mode="infer")
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_infer, num_workers=2)
    all_preds, all_ids = [], []
    letter_ids_tensor = torch.tensor(LETTER_IDS, device=DEVICE)
    NEG = torch.finfo(torch.float16).min

    for enc in tqdm(dl, desc="predict"):
        num_ch = enc.pop("num_choices")
        ids    = enc.pop("ids")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.amp.autocast("cuda", dtype=torch.float16):
            out = model(**enc)
        logits = out.logits
        last_idx = enc["attention_mask"].sum(dim=1) - 1
        gathered = logits[torch.arange(logits.size(0)), last_idx]
        letter_logits = gathered[:, letter_ids_tensor]
        mask = torch.arange(5, device=DEVICE).unsqueeze(0) >= num_ch.to(DEVICE).unsqueeze(1)
        letter_logits = letter_logits.masked_fill(mask, NEG)
        preds = letter_logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds); all_ids.extend(ids)
    model.train()
    return pd.DataFrame({"id": all_ids, "answer": all_preds})

@torch.no_grad()
def predict_tta(model, df, batch_size=4, n_perms=3):
    """Test-time augmentation: average letter logits across n_perms random choice orderings."""
    model.eval()
    letter_ids_tensor = torch.tensor(LETTER_IDS, device=DEVICE)
    NEG = torch.finfo(torch.float16).min

    # Collect logits across permutations, all remapped to ORIGINAL positions
    all_remapped = []   # list of (n_rows, 5) arrays
    base_ids, base_nch = None, None

    for perm_seed in range(n_perms):
        rng = random.Random(1000 + perm_seed)
        # Build a permuted copy of df
        df_perm = df.copy().reset_index(drop=True)
        perms = []
        for i, row in df_perm.iterrows():
            n = row["num_choices"]
            if perm_seed == 0:
                perm = list(range(n))   # identity for the first pass
            else:
                perm = list(range(n))
                rng.shuffle(perm)
            perms.append(perm)
            df_perm.at[i, "choices"] = [row["choices"][p] for p in perm]

        ds = VQADataset(df_perm, mode="infer")
        dl = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        collate_fn=collate_infer, num_workers=0)

        perm_logits = []
        ids_list, nch_list = [], []
        for enc in tqdm(dl, desc=f"TTA pass {perm_seed+1}/{n_perms}"):
            num_ch = enc.pop("num_choices")
            ids = enc.pop("ids")
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            with torch.amp.autocast("cuda", dtype=torch.float16):
                out = model(**enc)
            last_idx = enc["attention_mask"].sum(dim=1) - 1
            gathered = out.logits[torch.arange(out.logits.size(0)), last_idx]
            letter_logits = gathered[:, letter_ids_tensor].float().cpu().numpy()
            perm_logits.append(letter_logits)
            ids_list.extend(ids); nch_list.extend(num_ch.tolist())

        perm_logits = np.concatenate(perm_logits)   # (n_rows, 5)
        # REMAP: in this permutation, position-after-permutation `j` corresponds to
        # original position perms[i][j]. So to undo, scatter logits back to the original slot.
        remapped = np.full_like(perm_logits, fill_value=-1e9)
        for i, perm in enumerate(perms):
            for j, orig in enumerate(perm):
                remapped[i, orig] = perm_logits[i, j]
        all_remapped.append(remapped)
        if perm_seed == 0:
            base_ids, base_nch = ids_list, nch_list

    avg = np.mean(all_remapped, axis=0)
    mask = np.arange(5)[None, :] >= np.array(base_nch)[:, None]
    avg = np.where(mask, -1e9, avg)
    preds = avg.argmax(axis=1)
    return pd.DataFrame({"id": base_ids, "answer": preds.astype(int)})

def val_accuracy(model):
    preds = predict(model, val_df)
    merged = preds.merge(val_df[["id","answer"]], on="id", suffixes=("_pred","_gold"))
    return (merged.answer_pred == merged.answer_gold).mean(), preds

## 9. Visualization helpers

In [12]:
class TrainingLogger:
    """Persistent training log. Reloads from JSON on init so it survives Colab disconnects."""
    def __init__(self, log_path):
        self.path = Path(log_path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.data = {
            "step_metrics": [],
            "eval_metrics": [],
            "events": [],
            "meta": {
                "start_time": time.time(),
                "config": None,
                "version": None,
                "zero_shot_val_acc": None,
                "total_train_seconds": 0.0,
                "peak_gpu_mem_mb": 0.0,
                "trainable_params": None,
                "finished": False,
            },
            "final": {
                "val_breakdown": None,
                "test_pred_distribution": None,
                "best_val_acc": None,
                "best_val_step": None,
            },
        }
        if self.path.exists():
            try:
                with open(self.path) as f:
                    self.data = json.load(f)
                print(f"Loaded existing log ({len(self.data['step_metrics'])} steps, "
                      f"{len(self.data['eval_metrics'])} evals)")
            except Exception as e:
                print(f"Could not load log at {self.path}: {e}. Starting fresh.")

    def log_step(self, step, epoch, loss, lr, grad_norm, time_s, gpu_mem_mb):
        self.data["step_metrics"].append({
            "step": step, "epoch": epoch,
            "loss": float(loss), "lr": float(lr),
            "grad_norm": float(grad_norm), "time_s": float(time_s),
            "gpu_mem_mb": float(gpu_mem_mb),
        })
        self.data["meta"]["peak_gpu_mem_mb"] = max(
            self.data["meta"]["peak_gpu_mem_mb"], float(gpu_mem_mb))

    def log_eval(self, step, epoch, val_acc, breakdown=None):
        entry = {"step": step, "epoch": epoch, "val_acc": float(val_acc)}
        if breakdown is not None:
            entry["breakdown"] = breakdown
        self.data["eval_metrics"].append(entry)

    def log_event(self, step, event, detail=""):
        self.data["events"].append({
            "step": step, "event": event, "detail": detail, "time": time.time()})

    def update_meta(self, **kwargs):
        self.data["meta"].update(kwargs)

    def update_final(self, **kwargs):
        self.data["final"].update(kwargs)

    def save(self):
        tmp = self.path.with_suffix(".tmp.json")
        with open(tmp, "w") as f:
            json.dump(self.data, f, indent=2, default=str)
        tmp.replace(self.path)


def breakdown_metrics(val_df_with_gold, preds_df):
    """Slice val accuracy by num_choices / subject / grade / gold answer index."""
    m = preds_df.rename(columns={"answer": "pred"}).merge(
        val_df_with_gold[["id","answer","num_choices","subject","grade"]], on="id")
    m["correct"] = (m.pred == m.answer).astype(int)

    def by_col(col):
        g = m.groupby(col).agg(n=("correct","size"), acc=("correct","mean"))
        return {str(k): {"n": int(v["n"]), "acc": float(v["acc"])} for k, v in g.to_dict("index").items()}

    conf = {}
    for (gold, pred), cnt in m.groupby(["answer","pred"]).size().items():
        conf.setdefault(str(gold), {})[str(pred)] = int(cnt)

    return {
        "overall_acc": float(m.correct.mean()),
        "by_num_choices": by_col("num_choices"),
        "by_subject":     by_col("subject"),
        "by_grade":       by_col("grade"),
        "by_answer_idx":  by_col("answer"),
        "pred_distribution": {str(k): int(v) for k, v in m.pred.value_counts().sort_index().items()},
        "gold_distribution": {str(k): int(v) for k, v in m.answer.value_counts().sort_index().items()},
        "confusion": conf,
    }


LOG_PATH = OUT_DIR / "metrics.json"
logger = TrainingLogger(LOG_PATH)
logger.update_meta(config=CONFIG, version=VERSION)
logger.save()
print(f"Logger writing to {LOG_PATH}")

Logger writing to /content/drive/MyDrive/v10/outputs/metrics.json


## 10. Zero-shot baseline

In [13]:
acc0, _ = val_accuracy(model)
print(f"Zero-shot val accuracy: {acc0:.4f}  (floor to beat: 0.36 = predict-all-zeros)")
logger.update_meta(zero_shot_val_acc=float(acc0))
logger.log_event(0, "zero_shot", f"acc={acc0:.4f}")
logger.save()

predict:   0%|          | 0/262 [00:00<?, ?it/s]

Zero-shot val accuracy: 0.5754  (floor to beat: 0.36 = predict-all-zeros)


In [14]:
# Confirm prompt format isn't catastrophically wrong by sampling 3 prompts
print(f"Active prompt style: {CONFIG['prompt_style']}\n")
for i in [0, 100, 500]:
    row = train_df.iloc[i].to_dict()
    prompt = build_prompt(row, include_solution=False)
    print(f"--- train row {i} ({row['id']}) ---")
    print(f"prompt char length: {len(prompt)}")
    # show last 200 chars so we can see the model lands on "Answer:" correctly
    print(f"  ...prompt tail: {prompt[-200:]!r}")
    print()

Active prompt style: raw_image

--- train row 0 (train_07667) ---
prompt char length: 2511
  ...prompt tail: " the chances that ().\nChoices:\n  A. the male's tadpoles will be larger when they hatch\n  B. the male will carry his tadpoles through the forest\n  C. the male's tadpoles will become adult frogs\nAnswer:"

--- train row 100 (train_06457) ---
prompt char length: 2469
  ...prompt tail: ' square shows a cross between two guppies.\n\nQuestion: What is the probability that a guppy produced by this cross will have a golden body?\nChoices:\n  A. 4/4\n  B. 3/4\n  C. 1/4\n  D. 0/4\n  E. 2/4\nAnswer:'

--- train row 500 (train_05374) ---
prompt char length: 2251
  ...prompt tail: "ollowing could Shivani's test show?\nChoices:\n  A. if the new turbine could turn easily\n  B. whether the new turbine could produce 10% more electricity\n  C. how much the new turbine would weigh\nAnswer:"



## 11. Checkpoint Saving

In [15]:
from safetensors.torch import load_file as safe_load
from peft.utils import set_peft_model_state_dict   # <-- the missing import

def save_full_ckpt(path, model, optim, sched, scaler, epoch, batches_in_epoch, best_val):
    """Save LoRA adapter + optimizer + scheduler + scaler + counters + RNG."""
    p = Path(path); p.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(p))
    torch.save({
        "optim":  optim.state_dict(),
        "sched":  sched.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch":  epoch,
        "batches_in_epoch": batches_in_epoch,
        "best_val": best_val,
        "rng_torch":  torch.get_rng_state(),
        "rng_cuda":   torch.cuda.get_rng_state_all(),
        "rng_numpy":  np.random.get_state(),
        "rng_python": random.getstate(),
    }, p / "trainer_state.pt")

def _lora_fingerprint(model):
    """Frobenius norm of the first LoRA A matrix — a quick fingerprint for save/load consistency."""
    for n, p in model.named_parameters():
        if "lora_A" in n and p.requires_grad:
            return float(p.detach().float().norm().item())
    return float("nan")

def try_load_full_ckpt(path, model, optim, sched, scaler):
    """Returns (epoch, batches_in_epoch, best_val). (0, 0, -1.0) if no ckpt."""
    p = Path(path)
    state_file   = p / "trainer_state.pt"
    adapter_file = p / "adapter_model.safetensors"
    if not state_file.exists() or not adapter_file.exists():
        print(f"No checkpoint at {p} — starting fresh.")
        return 0, 0, -1.0

    print(f"Resuming from {p}")
    before = _lora_fingerprint(model)

    # PEFT-aware load: translates save-format keys → live-model keys (adapter name insertion)
    adapter_sd = safe_load(str(adapter_file))
    result = set_peft_model_state_dict(model, adapter_sd)
    # result.unexpected_keys should be empty for a matching adapter
    unexpected = getattr(result, "unexpected_keys", [])
    if unexpected:
        raise RuntimeError(f"unexpected keys after load: {unexpected[:3]}")

    after = _lora_fingerprint(model)
    print(f"  lora_A fingerprint: {before:.6f} -> {after:.6f}"
          + ("  (changed ✓)" if before != after else "  (UNCHANGED — load did nothing!)"))

    state = torch.load(state_file, map_location="cpu", weights_only=False)
    optim.load_state_dict(state["optim"])
    sched.load_state_dict(state["sched"])
    scaler.load_state_dict(state["scaler"])
    torch.set_rng_state(state["rng_torch"])
    torch.cuda.set_rng_state_all(state["rng_cuda"])
    np.random.set_state(state["rng_numpy"])
    random.setstate(state["rng_python"])
    print(f"  resumed at epoch={state['epoch']}, batches_in_epoch={state['batches_in_epoch']}, "
          f"best_val={state['best_val']:.4f}")
    return state["epoch"], state["batches_in_epoch"], state["best_val"]

## 12. EMA Helper

In [16]:
class LoRAEMA:
    def __init__(self, model, decay=0.999, warmup_steps=100):
        self.decay = decay
        self.warmup_steps = warmup_steps
        self.steps = 0
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone().float()

    @torch.no_grad()
    def update(self, model):
        self.steps += 1
        # During warmup: shadow tracks live weights exactly
        # After warmup: standard EMA
        if self.steps <= self.warmup_steps:
            for n, p in model.named_parameters():
                if p.requires_grad and n in self.shadow:
                    self.shadow[n].copy_(p.detach().float())
        else:
            for n, p in model.named_parameters():
                if p.requires_grad and n in self.shadow:
                    self.shadow[n].mul_(self.decay).add_(p.detach().float(), alpha=1 - self.decay)

    @torch.no_grad()
    def apply_to(self, model):
        backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                backup[n] = p.detach().clone()
                p.data.copy_(self.shadow[n].to(p.dtype))
        return backup

    @torch.no_grad()
    def restore(self, model, backup):
        for n, p in model.named_parameters():
            if n in backup:
                p.data.copy_(backup[n])

def val_accuracy_ema(model, ema):
    backup = ema.apply_to(model)
    try:
        acc, preds = val_accuracy(model)
    finally:
        ema.restore(model, backup)
    return acc, preds

ema = None # will be initialized inside train()

## 13. Train LoRA

In [17]:
from itertools import islice

def train(model, cfg, logger):
    train_ds = VQADataset(train_df_full, mode="train",
                      cot_prob=cfg["cot_prob"] if cfg["use_cot"] else 0.0)
    n_batches = len(train_ds) // cfg["batch_size"]
    steps_per_epoch = n_batches // cfg["grad_accum"]
    total_steps  = steps_per_epoch * cfg["epochs"]
    warmup_steps = int(total_steps * cfg["warmup_ratio"])

    optim  = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                               lr=cfg["lr"], weight_decay=0.0)
    sched  = get_cosine_schedule_with_warmup(optim, warmup_steps, total_steps)
    scaler = torch.amp.GradScaler("cuda")
    global ema
    ema = LoRAEMA(model, cfg["ema_decay"]) if cfg.get("use_ema", False) else None

    start_epoch, start_batches, best_val = try_load_full_ckpt(
        cfg["ckpt_latest_path"], model, optim, sched, scaler)
    if start_epoch > 0 or start_batches > 0:
        logger.log_event(len(logger.data["step_metrics"]),
                         "resumed", f"epoch={start_epoch}, batches={start_batches}")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.update_meta(trainable_params=int(trainable))

    model.train()
    global_step = sched.state_dict().get("_step_count", 0)
    run_start   = time.time()
    batch_timer = time.time()
    running     = 0.0

    for epoch in range(start_epoch, cfg["epochs"]):
        g = torch.Generator(); g.manual_seed(SEED + epoch)
        dl = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                        collate_fn=collate_train, num_workers=2, drop_last=True, generator=g)
        skip_n = start_batches if epoch == start_epoch else 0
        if skip_n: print(f"  skipping {skip_n} batches to resume")

        pbar = tqdm(islice(dl, skip_n, None),
                    desc=f"epoch {epoch+1}/{cfg['epochs']}",
                    initial=skip_n, total=len(dl))
        optim.zero_grad()
        batches_seen = skip_n

        for enc in pbar:
            batches_seen += 1
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            # Replace the loss computation block in cell 31 with:
            with torch.amp.autocast("cuda", dtype=torch.float16):
                letter_pos = enc.pop("_letter_pos")
                letter_id  = enc.pop("_letter_id")
                out = model(**enc)
                # Causal LM: logits[i, t] predicts token at position t+1.
                # To predict the letter at position letter_pos, read logits at letter_pos - 1.
                full_logits = out.logits[torch.arange(out.logits.size(0)), (letter_pos - 1).to(DEVICE)]
                # Restrict to the 5 letter-token logits — the only labels that exist for this task
                letter_ids_tensor = torch.tensor(LETTER_IDS, device=DEVICE)   # shape (5,)
                letter_logits = full_logits[:, letter_ids_tensor]              # (B, 5)
                # Map each batch row's letter_id to its index in [0..4]
                target_idx = torch.zeros(letter_id.size(0), dtype=torch.long, device=DEVICE)
                for i, lid in enumerate(letter_id.tolist()):
                    target_idx[i] = LETTER_IDS.index(lid)
                # Now smoothing distributes 10% across the 5 letters, not all 50k vocab tokens
                loss = F.cross_entropy(letter_logits.float(), target_idx,
                                      label_smoothing=cfg["label_smoothing"])
                loss = loss / cfg["grad_accum"]
            scaler.scale(loss).backward()
            running += loss.item() * cfg["grad_accum"]

            if batches_seen % cfg["grad_accum"] == 0:
                scaler.unscale_(optim)
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                prev_scale = scaler.get_scale()
                scaler.step(optim)
                scaler.update()
                # Only step the scheduler if optimizer.step() actually ran (no overflow)
                if scaler.get_scale() >= prev_scale:
                    sched.step()
                optim.zero_grad()
                if ema is not None:
                    ema.update(model)
                global_step += 1
                step_time = time.time() - batch_timer
                batch_timer = time.time()
                mem_mb = torch.cuda.max_memory_allocated() / 1e6

                step_loss = running / cfg["grad_accum"]
                logger.log_step(global_step, epoch,
                                loss=step_loss, lr=sched.get_last_lr()[0],
                                grad_norm=grad_norm.item(), time_s=step_time,
                                gpu_mem_mb=mem_mb)
                running = 0.0
                pbar.set_postfix(loss=f"{step_loss:.3f}",
                                 lr=f"{sched.get_last_lr()[0]:.2e}",
                                 gn=f"{grad_norm.item():.2f}")

                if global_step % cfg["ckpt_every_steps"] == 0:
                    save_full_ckpt(cfg["ckpt_latest_path"], model, optim, sched, scaler,
                                   epoch, batches_seen, best_val)
                    logger.save()

                if global_step % cfg["val_every_steps"] == 0:
                    if ema is not None:
                        acc, preds = val_accuracy_ema(model, ema)
                    else:
                        acc, preds = val_accuracy(model)
                    bd = breakdown_metrics(val_df, preds)
                    logger.log_eval(global_step, epoch, acc, breakdown=bd)
                    print(f"  step {global_step}: val={acc:.4f}")
                    if acc > best_val:
                        best_val = acc
                        model.save_pretrained(cfg["save_path"])
                        logger.log_event(global_step, "best_adapter_saved", f"val={acc:.4f}")
                        print(f"  ↑ saved best adapter (val={acc:.4f})")
                    logger.save()
                    model.train()

        acc, preds = (val_accuracy_ema(model, ema) if ema is not None else val_accuracy(model))
        bd = breakdown_metrics(val_df, preds)
        logger.log_eval(global_step, epoch, acc, breakdown=bd)
        print(f"epoch {epoch+1} end: val={acc:.4f}")
        if acc > best_val:
            best_val = acc
            model.save_pretrained(cfg["save_path"])
            logger.log_event(global_step, "best_adapter_saved", f"val={acc:.4f}")
        save_full_ckpt(cfg["ckpt_latest_path"], model, optim, sched, scaler,
                       epoch + 1, 0, best_val)
        logger.update_meta(total_train_seconds=
            logger.data["meta"]["total_train_seconds"] + (time.time() - run_start))
        logger.save()
        run_start = time.time()
        model.train()
        start_batches = 0

    logger.update_meta(finished=True)
    logger.save()
    return best_val

best_val = train(model, CONFIG, logger)
print(f"Best val accuracy: {best_val:.4f}")

No checkpoint at /content/drive/MyDrive/v10/outputs/ckpt_latest — starting fresh.


epoch 1/5:   0%|          | 0/1554 [00:00<?, ?it/s]

predict:   0%|          | 0/262 [00:00<?, ?it/s]

  step 50: val=0.6803
  ↑ saved best adapter (val=0.6803)


predict:   0%|          | 0/262 [00:00<?, ?it/s]

...

  step 900: val=0.7605
  ↑ saved best adapter (val=0.7605)


predict:   0%|          | 0/262 [00:00<?, ?it/s]

  step 950: val=0.7615
  ↑ saved best adapter (val=0.7615)


predict:   0%|          | 0/262 [00:00<?, ?it/s]

epoch 5 end: val=0.7634
Best val accuracy: 0.7634


## 14. Final val eval + test inference + submission.csv

In [18]:
# Final val accuracy with EMA + TTA (the numbers we trust)
backup = ema.apply_to(model) if ema is not None else None
try:
    val_acc, _ = val_accuracy(model)   # EMA-swapped if backup is set
    print(f"Final val accuracy (EMA, no TTA): {val_acc:.4f}")

    if CONFIG.get("tta_n_permutations", 1) > 1:
        val_tta_preds = predict_tta(model, val_df, n_perms=CONFIG["tta_n_permutations"])
        val_tta_acc = (val_tta_preds.merge(val_df[["id","answer"]], on="id",
                                            suffixes=("_pred","_gold"))
                       .apply(lambda r: r.answer_pred == r.answer_gold, axis=1).mean())
        print(f"Final val accuracy (EMA + TTA): {val_tta_acc:.4f}")

    # After the val_acc print:
    if CONFIG.get("use_train_plus_val", False):
        print("Note: model trained on train+val, so val acc here measures memorization, not generalization.")

finally:
    if ema is not None and backup is not None:
        ema.restore(model, backup)

predict:   0%|          | 0/262 [00:00<?, ?it/s]

Final val accuracy (EMA, no TTA): 0.7634


TTA pass 1/3:   0%|          | 0/262 [00:00<?, ?it/s]

TTA pass 2/3:   0%|          | 0/262 [00:00<?, ?it/s]

TTA pass 3/3:   0%|          | 0/262 [00:00<?, ?it/s]

Final val accuracy (EMA + TTA): 0.7634


In [19]:
# Test inference with EMA + TTA
backup = ema.apply_to(model) if ema is not None else None
try:
    n_perms = CONFIG.get("tta_n_permutations", 1)
    if n_perms > 1:
        print(f"Running test inference with TTA ({n_perms} permutations)...")
        test_preds = predict_tta(model, test_df, n_perms=n_perms)
    else:
        test_preds = predict(model, test_df, batch_size=4)

    sample = pd.read_csv(DATA_DRIVE / "sample_submission.csv")
    assert set(test_preds.columns) == {"id","answer"}, test_preds.columns
    assert set(test_preds.id) == set(sample.id), "ID set does not match sample_submission"
    test_preds = test_preds.set_index("id").loc[sample.id].reset_index()
    test_preds["answer"] = test_preds["answer"].astype(int)

    check = test_preds.merge(test_df[["id","num_choices"]], on="id")
    assert (check.answer < check.num_choices).all(), "some predictions exceed num_choices"

    sub_path = OUT_DIR / "submission.csv"
    test_preds.to_csv(sub_path, index=False)
    print(f"Wrote {sub_path}  ({len(test_preds)} rows)")

    # Final breakdown for the analyze notebook to read
    final_acc, final_preds = val_accuracy(model)
    final_bd = breakdown_metrics(val_df, final_preds)
finally:
    if ema is not None and backup is not None:
        ema.restore(model, backup)

evals = logger.data["eval_metrics"]
best_eval = max(evals, key=lambda e: e["val_acc"]) if evals else None

logger.update_final(
    val_breakdown=final_bd,
    test_pred_distribution={str(k): int(v) for k, v in test_preds.answer.value_counts().sort_index().items()},
    best_val_acc=best_eval["val_acc"] if best_eval else None,
    best_val_step=best_eval["step"]   if best_eval else None,
    final_val_acc=float(final_acc),
)
logger.save()

print(f"Final val acc: {final_acc:.4f}")
print("Answer distribution:", test_preds.answer.value_counts().sort_index().to_dict())
print(f"metrics.json at {logger.path}")

Running test inference with TTA (3 permutations)...


TTA pass 1/3:   0%|          | 0/252 [00:00<?, ?it/s]

TTA pass 2/3:   0%|          | 0/252 [00:00<?, ?it/s]

TTA pass 3/3:   0%|          | 0/252 [00:00<?, ?it/s]

Wrote /content/drive/MyDrive/v10/outputs/submission.csv  (1008 rows)


predict:   0%|          | 0/262 [00:00<?, ?it/s]

Final val acc: 0.7634
Answer distribution: {0: 337, 1: 356, 2: 230, 3: 80, 4: 5}
metrics.json at /content/drive/MyDrive/v10/outputs/metrics.json


In [20]:
# val-set predictions as JSON for inspection
val_pred_dump = []
final_acc, final_preds = val_accuracy(model)
for _, row in val_df.merge(final_preds.rename(columns={"answer":"pred"}), on="id").iterrows():
    if row["pred"] != row["answer"]:                        # only the wrong ones
        val_pred_dump.append({
            "id": row["id"],
            "subject": row["subject"],
            "num_choices": int(row["num_choices"]),
            "gold": int(row["answer"]),
            "pred": int(row["pred"]),
            "question": row["question"][:300],
            "choices": row["choices"],
        })
with open(OUT_DIR / "val_errors.json", "w") as f:
    json.dump(val_pred_dump, f, indent=2)
print(f"Wrote {len(val_pred_dump)} val errors")

predict:   0%|          | 0/262 [00:00<?, ?it/s]

Wrote 222 val errors


## Notes

- **Per-version workflow**: duplicate the notebook in `Colab Notebooks/`, change `VERSION` at the top, and run. Adapter + `submission.csv` go to `MyDrive/{VERSION}/outputs/`, isolated from other versions.
- **Image cache lifecycle**: `ensure_local_images()` is a no-op after the first run within a Colab session. When the VM recycles (disconnect or ~12h), the sentinel and extracted images disappear and it runs again. CSV reads always come from Drive — they're small.
- **Kaggle offline submission**: Kaggle's final eval has no internet. To port: upload base SmolVLM weights as a Kaggle Dataset, upload your best LoRA adapter as another Dataset, and change `model_id` / adapter load path to those local Kaggle paths.
- **If Colab OOMs**: drop `image_longest_edge` to 336, set `batch_size=1` and `grad_accum=16`, or cut `max_text_tokens` to 768.
- **Ideas for later versions**: weighted sampling on the answer-index imbalance; ensemble plain + CoT adapters by averaging letter logits before masked argmax; swap LoRA targets to just `q_proj, v_proj` and push rank to 32 within the 5M cap.